In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from xgboost import XGBClassifier
df = pd.read_csv(r"C:\Users\Alirexa\Desktop\aitech\bank_fraud2.csv")
df = df.sample(n=100000,random_state=42)
df = df.drop(columns=[
        "transaction_id",
        "customer_id",
        "transaction_date",
        "transaction_time",
        "fraud_type"],
    errors="ignore")
y = df["is_fraud"]
X = df.drop("is_fraud",axis=1)

X = pd.get_dummies(
    X,
    drop_first=True,
    dtype="int8"
)
print("Shape After Encoding:")
print(X.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
lr_model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    solver="saga"
)
lr_model.fit(
    X_train,
    y_train
)
y_pred_lr = lr_model.predict(
    X_test
)
print(" LOGISTIC REGRESSION ")
print(
    classification_report(
        y_test,
        y_pred_lr,
        zero_division=0
    )
)
#random forest
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=10,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)
rf_model.fit(
    X_train,
    y_train
)
y_prob_rf = rf_model.predict_proba(
    X_test
)[:, 1]
rf_threshold = 0.40
y_pred_rf = (
    y_prob_rf > rf_threshold
).astype(int)
print("\n RANDOM FOREST ")
print(
    classification_report(
        
        y_test,
        y_pred_rf,
        zero_division=0
    )
)
#xgb
scale_pos_weight = (
    y_train.value_counts()[0]
    /
    y_train.value_counts()[1]
)
xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)
xgb_model.fit(
    X_train,
    y_train
)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
xgb_threshold = 0.40
y_pred_xgb = (
    y_prob_xgb > xgb_threshold
).astype(int)
print(" XGBOOST ")
print(classification_report(y_test,y_pred_xgb,zero_division=0))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    precision_recall_curve,
    f1_score
)

#Fraud distribution
plt.figure(figsize=(7,5))
ax=sns.countplot(x="is_fraud",data=df)
for c in ax.containers:
    ax.bar_label(c)
plt.title("Fraud Distribution")
plt.show()

#Heatmap
corr=df.select_dtypes(include=["int64","float64"]).corr()
mask=np.triu(np.ones_like(corr,dtype=bool))
plt.figure(figsize=(14,10))
sns.heatmap(corr,mask=mask,cmap="RdBu_r",annot=True,fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

#Histogram
plt.figure(figsize=(8,5))
sns.histplot(data=df,x="transaction_amount",hue="is_fraud",bins=40,kde=True)
plt.show()

#Feature importance
imp=pd.DataFrame({
    "Feature":X.columns,
    "Importance":xgb_model.feature_importances_
}).sort_values("Importance",ascending=False).head(15)

plt.figure(figsize=(10,7))
sns.barplot(data=imp,x="Importance",y="Feature")
plt.title("Top Features")
plt.show()

#Confusion Matrix
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_xgb,
    cmap="Blues",
    values_format="d"
)
plt.title("Confusion Matrix")
plt.show()

#ROC
fpr,tpr,_=roc_curve(y_test,y_prob_xgb)
roc_auc=auc(fpr,tpr)
plt.figure(figsize=(7,6))
plt.plot(fpr,tpr,label=f"AUC={roc_auc:.3f}")
plt.plot([0,1],[0,1],'--')
plt.legend()
plt.grid()
plt.title("ROC Curve")
plt.show()

#Compare models
scores=[
    f1_score(y_test,y_pred_lr),
    f1_score(y_test,y_pred_rf),
    f1_score(y_test,y_pred_xgb)
]
names=["Logistic","Random Forest","XGBoost"]

plt.figure(figsize=(8,5))
bars=plt.bar(names,scores)
plt.bar_label(bars,fmt="%.3f")
plt.ylim(0,1)
plt.title("F1 Comparison")
plt.show()